# Phase 0 -- Helium Emotion Representation Validation

**PersonaPlex in-house emotion network project** -- this notebook answers one question *before any training*:

> **Can PersonaPlex's own internal representations (Helium main-LM hidden states) predict reply emotion well enough to train a lightweight network on them -- and which representation input + head family is best?**

## What this notebook does

1. Verifies the environment, clones/uses the repository, installs dependencies.
2. Programmatically inspects the PersonaPlex/Helium architecture (joint text+audio bottleneck, 12.5 Hz step rate) straight from the repository source -- no assumptions.
3. Downloads (resumably, with checksums) a **configurable subset** of the existing PersonaPlex reply dataset (`niloy629/hdtf_preprocess`, ~23 GB, ~6,890 aligned 8-second samples) and verifies **per-sample time alignment** between Helium hidden states and reply audio (duration structure, energy cross-correlation diagnostics, and the trained adapter regression).
4. Runs **two offline teacher models**:
   - **Vocal-expression teacher** (`emotion2vec_plus_base`, the same model family the live IMTalker emotion loop uses on the user microphone) -> what emotion the reply *sounds like*.
   - **Semantic-content teacher** (ASR + text classifier) -> what emotion the reply *says*.
   Both label sets plus full teacher score vectors are **preserved separately**; disagreements and low-confidence rows go to a **human review queue** (`corrections.csv`).
5. Trains and evaluates a **representation menu** of candidate network inputs (Helium hidden only, hidden + Wav2Vec adapter targets, hidden + Mimi codes, hidden + acoustic stats) and head families (linear probe on pooled features; GRU / tiny Transformer on per-step sequences) against **three targets separately: vocal, semantic, expressed** -- so we measure exactly which information the representations carry.
6. Measures latency, VRAM, parameter counts, and calibration (ECE) of every candidate head.
7. Writes `phase0_results.json` + `phase0_report.md` with an explicit recommendation consumed by Phase 2.

## Key principles (enforced throughout)

- **Offline teachers are labeling tools only.** The final live system runs *one* lightweight head on PersonaPlex's own hidden states -- no teacher runs in the live path.
- **No dataset construction, no FM/renderer, no user-voice inputs** in this notebook. Phase 1 collects the PersonaPlex dataset; the existing IMTalker conditioning path is reused unchanged downstream.
- **Ground truth = expressed emotion** (what the model actually produced), with intended/scenario emotion kept as metadata only -- the mapping from intended to expressed is a distribution, not a function.
- **Alignment is verified per sample** (duration consistency + energy cross-correlation), never assumed from shapes.
- **Everything resumable**: downloads, extraction, labeling, feature extraction, and probe training all checkpoint to disk; re-running a cell skips completed work.

## Hardware / time / storage (RunPod Jupyter, subset mode defaults)

| Item | Requirement |
|---|---|
| GPU | 1 x CUDA GPU, >= 24 GB VRAM recommended (teachers only need ~3 GB; 16 GB works) |
| Disk | subset mode: ~8 GB (1-2 archive parts + extracted subset); full mode: ~60 GB (23 GB archives + extracted) |
| RAM | >= 16 GB |
| Time (subset, ~600 samples) | ~30-60 min download+extract, ~10-20 min labeling, ~5-15 min probes |

## Outputs

| Artifact | Path |
|---|---|
| Results JSON (all metrics + recommendation) | `/workspace/persona_emotion/phase0/results/phase0_results.json` |
| Human-readable report | `/workspace/persona_emotion/phase0/phase0_report.md` |
| Dual teacher labels + fusion | `/workspace/persona_emotion/phase0/labels.csv` |
| Human review queue / corrections | `review_queue.csv` / `corrections.csv` |
| Alignment verification | `valid_index.csv`, `plots/alignment_*.png` |
| Probe artifacts | `/workspace/persona_emotion/phase0/probes/` |

**Run this notebook top-to-bottom first.** Only after its report shows which representation carries which emotion information should you proceed to Phase 1 (dataset collection).

In [ ]:
# ============================ Configuration =================================
# All paths and knobs are configurable here. Defaults target a RunPod Jupyter
# pod where this notebook runs either from inside the repository (recommended:
# clone the repo first, or upload it) or standalone (it will clone itself).

import os, sys, json, time, math, random, hashlib, subprocess, shutil, logging
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/MoshiHead/IMTalker_Personaplex_emotion_v1_auto_emotion_from_helium.git"

WORKSPACE = Path(os.environ.get("WORKSPACE", "/workspace"))
PROJECT_ROOT = Path(os.environ.get("SPEECH2AVATAR_ROOT", str(WORKSPACE / "speech2avatar")))
PHASE0_DIR = Path(os.environ.get("PERSONA_EMO_PHASE0_DIR", str(WORKSPACE / "persona_emotion" / "phase0")))

# --- Dataset (existing PersonaPlex reply dataset on the HF hub) ---
DATASET_HF_REPO = "niloy629/hdtf_preprocess"
DATASET_SUBDIR = "personaplex_helium_w2v_frontend_adapter/dataset"
DATASET_ARCHIVE_STEM = "personaplex_helium_w2v_adapter_dataset.tar.zst"
DATASET_EXTRACT_DIR = Path(os.environ.get("PERSONA_EMO_DATASET_DIR", str(WORKSPACE / "personaplex_frontend_adapter_dataset")))
FULL_DATASET = False            # True: extract all ~6,890 samples (~23 GB download, ~60 GB disk)
MAX_SAMPLES = 600               # used when FULL_DATASET is False (subset mode)
KEEP_PARTS = False              # keep downloaded .part files after extraction
VERIFY_CHECKSUMS = True         # verify each part against SHA256SUMS before extracting

# --- Offline teacher models (labeling only; never part of the live system) ---
AUDIO_TEACHER_MODEL = "emotion2vec/emotion2vec_plus_base"   # vocal expression
TEXT_TEACHER_MODEL = "j-hartmann/emotion-english-distilroberta-base"  # semantic content
ASR_MODEL = "openai/whisper-tiny.en"                        # transcripts for the text teacher
RUN_TEXT_TEACHER = True         # set False to skip ASR + text teacher (faster, vocal-only)
TEACHER_DEVICE = "cuda"         # "cpu" avoids GPU contention (slower)

# --- Emotion label contract (must match generator/FM.py EMOTION_MAP) ---
EMOTION_NAMES = {0: "neutral", 1: "happy", 2: "sad", 3: "angry", 4: "surprise", 5: "fear"}
EMOTION_IDS = {v: k for k, v in EMOTION_NAMES.items()}
NUM_EMOTIONS = 6
DISGUST_MAP = 3   # teacher disgust -> avatar class: 3=angry (visceral negative), 0=neutral alt

# --- Probes / evaluation ---
SEED = 1234
PCA_COMPONENTS = 30
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
RUN_MIMI_CODES = True   # re-encode reply WAV through Mimi to test audio-token-only prediction;
                        # needs the gated nvidia/personaplex-7b-v1 Mimi asset + HF_TOKEN, else auto-disabled
RUN_TORCH_PROBES = True # train small GRU/Transformer probes on per-step hidden sequences
TORCH_PROBE_EPOCHS = 25

# --- Derived paths ---
RESULTS_DIR = PHASE0_DIR / "results"
PLOTS_DIR = PHASE0_DIR / "plots"
PROBES_DIR = PHASE0_DIR / "probes"
FEAT_DIR = PHASE0_DIR / "features"
LABELS_CSV = PHASE0_DIR / "labels.csv"
REVIEW_QUEUE_CSV = PHASE0_DIR / "review_queue.csv"
CORRECTIONS_CSV = PHASE0_DIR / "corrections.csv"
VALID_INDEX_CSV = PHASE0_DIR / "valid_index.csv"
RESULTS_JSON = RESULTS_DIR / "phase0_results.json"
REPORT_MD = PHASE0_DIR / "phase0_report.md"
STATE_JSON = PHASE0_DIR / "state.json"

for _p in (PHASE0_DIR, RESULTS_DIR, PLOTS_DIR, PROBES_DIR, FEAT_DIR):
    _p.mkdir(parents=True, exist_ok=True)

def log(msg):
    line = "[%s] %s" % (time.strftime("%H:%M:%S"), msg)
    print(line, flush=True)
    with open(PHASE0_DIR / "phase0.log", "a", encoding="utf-8") as f:
        f.write(line + "\n")

def save_json(obj, path):
    tmp = str(path) + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)

random.seed(SEED)
import numpy as np
np.random.seed(SEED)

log("Configuration loaded.")
log("PROJECT_ROOT = %s" % PROJECT_ROOT)
log("PHASE0_DIR   = %s" % PHASE0_DIR)
log("Dataset mode = %s" % ("FULL (~6,890 samples, ~23 GB)" if FULL_DATASET else "SUBSET (max %d samples)" % MAX_SAMPLES))
log("Text teacher = %s | Mimi codes = %s" % ("ON" if RUN_TEXT_TEACHER else "OFF", "ON" if RUN_MIMI_CODES else "OFF"))

In [ ]:
# =========================== Environment inspection =========================
def run_cmd(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=90).stdout.strip()
    except Exception as e:
        return "<error: %s>" % e

log("Python : " + sys.version.split()[0])
log("CWD    : " + os.getcwd())

_gpu_name, _vram_gb = "none", 0.0
import torch
if torch.cuda.is_available():
    _gpu_name = torch.cuda.get_device_name(0)
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    log("GPU    : %s (%.1f GB VRAM)" % (_gpu_name, _vram_gb))
    log("torch  : %s | CUDA %s" % (torch.__version__, torch.version.cuda))
else:
    raise RuntimeError("No CUDA GPU detected. The offline teacher models need a GPU (or set TEACHER_DEVICE='cpu' and expect slow labeling).")

_disk = shutil.disk_usage(str(WORKSPACE if WORKSPACE.exists() else Path.cwd()))
log("Disk   : %.1f GB free on workspace volume" % (_disk.free / 1e9))
_mem_gb = 0.0
try:
    with open("/proc/meminfo") as _f:
        _mem_gb = int(_f.readline().split()[1]) / 1e6
except Exception:
    pass
if _mem_gb:
    log("RAM    : %.1f GB" % _mem_gb)

_need_gb = 70 if FULL_DATASET else 12
assert _disk.free / 1e9 > _need_gb, "Not enough free disk for the chosen dataset mode (need ~%d GB)." % _need_gb
log("Environment OK.")

In [ ]:
# ========================= Repository acquisition ===========================
# Idempotent: uses the in-place repository when this notebook already runs
# inside it; otherwise clones from GitHub. Validates the expected structure.
REPO_MARKER_FILES = [
    "IMTalker/liveTry.py",
    "IMTalker/auto_emotion.py",
    "personaplex/moshi/moshi/models/lm.py",
    "IMTalker/requirement.txt",
]

def looks_like_repo(p):
    p = Path(p)
    return p.is_dir() and all((p / f).exists() for f in REPO_MARKER_FILES)

_in_place = None
for _cand in [Path.cwd(), *Path.cwd().parents]:
    if looks_like_repo(_cand):
        _in_place = _cand
        break

if _in_place is None and looks_like_repo(PROJECT_ROOT):
    _in_place = PROJECT_ROOT

if _in_place is not None:
    PROJECT_ROOT = _in_place
    log("Using in-place repository: %s" % PROJECT_ROOT)
else:
    if PROJECT_ROOT.exists() and looks_like_repo(PROJECT_ROOT):
        log("Using existing repository at %s" % PROJECT_ROOT)
    else:
        log("Cloning %s -> %s" % (GITHUB_REPO_URL, PROJECT_ROOT))
        _r = subprocess.run("git clone --depth 1 %s %s" % (GITHUB_REPO_URL, str(PROJECT_ROOT)),
                            shell=True, capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("git clone failed: %s\nIf the GitHub repository is not accessible yet, copy this notebook into your local repository clone and run it from there." % _r.stderr[-1500:])
        if not looks_like_repo(PROJECT_ROOT):
            raise RuntimeError("Repository cloned but the expected structure is missing. If the GitHub repo is still empty, push your local code first, or copy this notebook into your local repository and run it from there.")

REPO_COMMIT = run_cmd("git -C %s rev-parse HEAD" % str(PROJECT_ROOT)) or "unknown"
IMTALKER_DIR = PROJECT_ROOT / "IMTalker"
assert (IMTALKER_DIR / "liveTry.py").exists()
log("Repository ready (commit %s)" % REPO_COMMIT[:12])

In [ ]:
# ============================ Dependencies ==================================
# Minimal, targeted install for Phase 0 (we import auto_emotion.py for the
# vocal teacher and moshi.models.* for architecture inspection / optional
# Mimi re-encoding -- NOT the renderer/generator stack, so the full
# IMTalker/requirement.txt is unnecessary here). Idempotent: already-installed
# packages are detected and skipped.

import shlex

def pip_install(pkg_spec):
    # shell=False + arg list: version specifiers like sphn>=0.2.0,<0.3.0 must
    # never go through a shell (the '<' would be read as input redirection).
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + shlex.split(pkg_spec)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2500:])
        print(r.stderr[-2500:])
        raise RuntimeError("pip install failed: %s" % pkg_spec)
    log("pip ok: %s" % pkg_spec)

def have(mod):
    r = subprocess.run([sys.executable, "-c", "import %s" % mod], capture_output=True)
    return r.returncode == 0

for _m, _pkg in [
    ("sklearn", "scikit-learn==1.3.2"),
    ("soundfile", "soundfile"),
    ("zstandard", "zstandard"),
    ("safetensors", "safetensors"),
    ("einops", "einops"),
    ("sphn", "sphn>=0.2.0,<0.3.0"),
    ("sentencepiece", "sentencepiece"),
    ("tqdm", "tqdm"),
]:
    if not have(_m):
        pip_install(_pkg)
if not have("funasr"):
    pip_install("funasr")
# Re-assert compatibility pins AFTER funasr (it resolves unpinned deps).
# NOTE: transformers is deliberately NOT pinned. The IMTalker live-server pin
# (transformers==4.30.2) cannot install on Python 3.12: its tokenizers<0.14
# constraint has no cp312 wheel, so pip attempts a Rust source build and fails.
# Phase 0 does not run the live-server stack; the transformers that funasr
# resolved is what ASR + the text teacher use.
# matplotlib floor 3.10.1: earlier 3.x crashes under IPython 9 (their
# install_repl_displayhook imports IPython.core.pylabtools.backend2gui, removed
# in IPython 9 -> ImportError on the first figure). 3.10.x still supports numpy 1.26.
pip_install("\"numpy<2.0.0\" \"pandas<2.2.0\" \"matplotlib>=3.10.1\"")

import numpy, pandas, sklearn, soundfile, matplotlib, tqdm
import importlib.metadata as _imd
log("numpy %s | pandas %s | sklearn %s | transformers %s" % (
    numpy.__version__, pandas.__version__, sklearn.__version__,
    _imd.version("transformers")))
log("All dependencies ready.")

In [ ]:
# ================== PersonaPlex architecture inspection =====================
# Programmatic verification of WHAT the Helium hidden states are, straight
# from the repository source. No model weights are loaded in this cell.
_moshi_pkg = PROJECT_ROOT / "personaplex" / "moshi"
if str(_moshi_pkg) not in sys.path:
    sys.path.insert(0, str(_moshi_pkg))

import moshi.models.lm as moshi_lm
import moshi.models.loaders as moshi_loaders
import inspect

_kw = dict(moshi_loaders._lm_kwargs)
_kw["dep_q"] = 16  # get_moshi_lm() overrides this
print("Helium main LM config (moshi.models.loaders):")
for _k in ("dim", "num_layers", "num_heads", "text_card", "card", "n_q", "dep_q", "context", "max_period"):
    print("  %-12s = %s" % (_k, _kw.get(_k)))
print("  delays       = %s (length %d = 1 text stream + 16 audio codebooks)" % (_kw["delays"], len(_kw["delays"])))
print()
print("=> d_model = 4096, context = 3000 steps. At Mimi rate 12.5 Hz:")
print("   8-second sample = 100 steps; full context window = 3000 steps = 240 s.")

print()
print("embed_codes (the joint text+audio bottleneck):")
print(inspect.getsource(moshi_lm.LMModel.embed_codes))
print("forward_embeddings (transformer -> out_norm; hidden state BEFORE text_linear):")
print(inspect.getsource(moshi_lm.LMModel.forward_embeddings))
print("=> The 4096-dim hidden state is the ONLY representation through which the")
print("   text token and all audio codebooks jointly influence the next step.")
print("   Emotion conditioning must therefore be readable from it (or from the")
print("   tokens/codes that feed it).")

print()
print("Live capture contract (AH server):")
_ah = IMTALKER_DIR / "liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py"
_ah_lines = _ah.read_text(encoding="utf-8").splitlines()
for _i, _l in enumerate(_ah_lines):
    if ("personaplex_step_with_hidden" in _l or "helium_hidden = layer_hidden[:1, -1:]" in _l
            or "reply_codes = tokens[:, 1:]" in _l or "MAIN_CODEBOOKS" in _l):
        print("  L%-4d %s" % (_i + 1, _l.strip()))
print()
print("=> Live system captures layer[-2] (post-transformer, pre-out_norm) hidden")
print("   states at 12.5 Hz and 8 audio codebooks per step. Phase 0 features use")
print("   the same contract; Phase 1 replicates this capture exactly.")

print()
_ser = IMTALKER_DIR / "auto_emotion.py"
assert _ser.exists(), "auto_emotion.py (vocal teacher) not found in the repository"
print("Vocal teacher class found: IMTalker/auto_emotion.py::SpeechEmotionClassifier")

In [ ]:
# ==================== Dataset download + streaming extraction ===============
# The dataset is ONE tar.zst archive that was compressed first and then
# byte-split into 6 parts (~4 GB each, ~23 GB total). Consequence: a single
# part is a TRUNCATED stream -- extraction must read the parts concatenated
# in order. Design:
#   * parts are downloaded one at a time (resumable, each SHA256-verified)
#   * extraction streams the concatenation of all parts downloaded so far
#   * a truncated tail (parts not yet downloaded) is tolerated: the stream
#     stops cleanly, the loop downloads the next part and re-extracts
#   * subset mode stops as soon as MAX_SAMPLES helium tensors are out --
#     remaining parts are never downloaded if the target is reached early
#   * already-written members are skipped by exact size match (this also
#     auto-repairs any truncated file left by an interrupted run)
from huggingface_hub import hf_hub_download, list_repo_files
import zstandard as zstd
import tarfile

WANTED_DIRS = ("helium", "reply_wav_24k", "w2v_final_50hz", "w2v_frontend_50hz", "stats")
TARGET_N = 10**9 if FULL_DATASET else MAX_SAMPLES

_state = json.loads(STATE_JSON.read_text()) if STATE_JSON.exists() else {}
if _state.get("format") != "v2":
    if _state:
        log("legacy download state detected -- resuming with size-verified skips (truncated files auto-repair)")
    _state = {"format": "v2", "extracted": int(_state.get("extracted", 0))}

def sha256_file(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def map_member(name):
    segs = [s for s in name.split("/") if s]
    for i, seg in enumerate(segs):
        if seg in WANTED_DIRS:
            return Path(seg).joinpath(*segs[i + 1:])
    return None

def load_sums():
    sums_path = hf_hub_download(DATASET_HF_REPO, DATASET_SUBDIR + "/SHA256SUMS", repo_type="dataset")
    sums = {}
    with open(sums_path) as f:
        for line in f:
            parts = line.split()
            if len(parts) == 2:
                sums[parts[1].strip()] = parts[0].strip()
    return sums

class ChainedReader:
    # file-like read() over a sequence of part files, in order: the split
    # archive only decompresses correctly as this concatenation.
    def __init__(self, paths):
        self.paths = list(paths)
        self._i = 0
        self._fh = None
    def _advance(self):
        if self._fh is not None:
            self._fh.close()
            self._fh = None
        if self._i >= len(self.paths):
            return False
        self._fh = open(self.paths[self._i], "rb")
        self._i += 1
        return True
    def read(self, size=-1):
        out = []
        need = size
        while size < 0 or need > 0:
            if self._fh is None and not self._advance():
                break
            data = self._fh.read(-1 if size < 0 else need)
            if not data:
                self._fh.close()
                self._fh = None
                continue
            out.append(data)
            if size > 0:
                need -= len(data)
        return b"".join(out)
    def close(self):
        if self._fh is not None:
            self._fh.close()
            self._fh = None

_counters = {"extracted": int(_state.get("extracted", 0)), "skipped": 0}

def extract_stream(part_paths):
    # returns "target" | "complete" | "truncated"
    reader = ChainedReader(part_paths)
    try:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(reader) as zr:
            with tarfile.open(fileobj=zr, mode="r|") as tar:
                for member in tar:
                    rel = map_member(member.name)
                    if rel is None:
                        continue
                    out = DATASET_EXTRACT_DIR / rel
                    if member.isdir():
                        out.mkdir(parents=True, exist_ok=True)
                        continue
                    if out.exists() and out.stat().st_size == member.size:
                        _counters["skipped"] += 1
                        continue
                    out.parent.mkdir(parents=True, exist_ok=True)
                    tmp_out = out.with_name(out.name + ".partial")
                    src = tar.extractfile(member)
                    with open(tmp_out, "wb") as dst:
                        shutil.copyfileobj(src, dst, length=1 << 22)
                    os.replace(tmp_out, out)
                    if rel.parts and rel.parts[0] == "helium":
                        _counters["extracted"] += 1
                        if _counters["extracted"] % 100 == 0:
                            log("  extracted %d samples..." % _counters["extracted"])
                        if _counters["extracted"] >= TARGET_N:
                            return "target"
        return "complete"
    except (tarfile.ReadError, EOFError, zstd.ZstdError) as e:
        log("  stream reached the truncation point (%s) -- fetching more parts" % type(e).__name__)
        return "truncated"
    finally:
        reader.close()

# clean stale .partial files from any previously interrupted run
for _p in DATASET_EXTRACT_DIR.rglob("*.partial"):
    _p.unlink()

_sums = load_sums() if VERIFY_CHECKSUMS else {}
_parts = sorted(f for f in list_repo_files(DATASET_HF_REPO, repo_type="dataset")
                if f.startswith(DATASET_SUBDIR + "/" + DATASET_ARCHIVE_STEM + ".part-"))
log("Dataset has %d archive parts on the hub (single split tar.zst -- extracted as one concatenated stream)." % len(_parts))

_local_parts = []
_result = "truncated"
for _part in _parts:
    _short = _part.split("/")[-1]
    log("ensuring part %s ..." % _short)
    _t0 = time.time()
    _local = hf_hub_download(DATASET_HF_REPO, _part, repo_type="dataset")
    if VERIFY_CHECKSUMS and _short in _sums:
        _h = sha256_file(_local)
        if _h != _sums[_short]:
            Path(_local).unlink(missing_ok=True)
            raise RuntimeError("Checksum mismatch for %s; deleted. Re-run this cell to re-download." % _short)
    log("  part ready (%.0f s)" % (time.time() - _t0))
    _local_parts.append(_local)
    _result = extract_stream(_local_parts)
    _state["extracted"] = _counters["extracted"]
    save_json(_state, STATE_JSON)
    log("  extracted=%d skipped=%d (%s)" % (_counters["extracted"], _counters["skipped"], _result))
    if _result in ("target", "complete"):
        break

if _result == "truncated" and not FULL_DATASET and _counters["extracted"] < TARGET_N:
    raise RuntimeError("Ran out of parts before reaching the subset target -- helium/ may be ordered late in the archive; set FULL_DATASET=True.")

if not KEEP_PARTS:
    for _lp in _local_parts:
        Path(_lp).unlink(missing_ok=True)
    log("part files removed from the HF cache (KEEP_PARTS=False)")

log("Extraction finished: %d helium samples available (%s)." % (_counters["extracted"], _result))

## Pairing and per-sample alignment verification

The dataset pairs `helium/<id>.pt` hidden-state tensors with `reply_wav_24k/<id>.wav` decoded reply audio (plus Wav2Vec adapter targets). **We never assume alignment from shapes.** For every paired sample we verify:

1. **Duration consistency** -- hidden steps * 80 ms must match the WAV duration (10% tolerance).
2. **Energy alignment** -- the per-step RMS energy envelope of the reply audio must co-vary with the per-step hidden-state norms at **zero lag** (normalized cross-correlation argmax within +/-2 steps; the Mimi delays config shifts some codebooks by 1 step, so lag 0 or +1 is expected).
3. **Voiced mask** -- per-step voiced/unvoiced decisions used later for masked pooling and probe evaluation.

Samples failing verification are flagged (not silently dropped) and reported with reasons.

In [ ]:
# ================ Pairing + alignment verification ===========================
import soundfile as sf
from tqdm.auto import tqdm

STEP_SEC = 0.08  # 12.5 Hz Mimi frame rate (1920 samples at 24 kHz)

def stem_key(p):
    n = p.name
    for suf in (".pt", ".wav"):
        if n.endswith(suf):
            n = n[:-len(suf)]
    for tag in ("_helium", "_w2v_final", "_w2v_frontend", "_reply_wav_24k"):
        n = n.replace(tag, "")
    return n

def index_dir(subdir, suffix):
    d = DATASET_EXTRACT_DIR / subdir
    if not d.is_dir():
        return {}
    return {stem_key(p): p for p in sorted(d.glob("*" + suffix))}

_helium = index_dir("helium", ".pt")
_wavs = index_dir("reply_wav_24k", ".wav")
_adapt_final = index_dir("w2v_final_50hz", ".pt")
_adapt_front = index_dir("w2v_frontend_50hz", ".pt")
log("found: helium=%d wav=%d adapter_final=%d adapter_frontend=%d" % (
    len(_helium), len(_wavs), len(_adapt_final), len(_adapt_front)))
_paired = sorted(set(_helium) & set(_wavs))
log("paired helium<->wav: %d (unmatched helium: %d, unmatched wav: %d)" % (
    len(_paired), len(set(_helium) - set(_wavs)), len(set(_wavs) - set(_helium))))
assert len(_paired) > 0, "No paired samples found -- check the extraction step."

def load_helium(path):
    # The dataset's own tooling (IMTalker/tools/*.py) reads these files with keys
    # "helium_states" / "helium"; extra aliases + a generic fallback keep this
    # robust against future repacks.
    h = torch.load(path, map_location="cpu")
    if isinstance(h, dict):
        t = None
        for k in ("helium_states", "helium", "full_helium_raw", "hidden", "hidden_states", "last_hidden_state"):
            if torch.is_tensor(h.get(k)):
                t = h[k]
                break
        if t is None:
            cands = [v for v in h.values() if torch.is_tensor(v) and v.ndim >= 2 and v.shape[-1] == 4096]
            if not cands:
                cands = [v for v in h.values() if torch.is_tensor(v)]
            if len(cands) == 1:
                t = cands[0]
        if t is None:
            raise RuntimeError("no helium tensor found in %s; dict keys: %s" % (path, list(h.keys())))
        h = t
    h = h.float() if h.dtype in (torch.float16, torch.bfloat16) else h
    if h.ndim == 3 and h.shape[0] == 1:
        h = h[0]
    if h.ndim == 3 and h.shape[-1] == 1:
        h = h[..., 0]
    assert h.ndim == 2 and h.shape[1] == 4096, "unexpected helium tensor shape %s in %s" % (tuple(h.shape), path)
    return h  # [T, 4096]

def xcorr_lag(a, b, max_lag=3):
    a = (a - a.mean()) / (a.std() + 1e-8)
    b = (b - b.mean()) / (b.std() + 1e-8)
    best_lag, best_c = 0, -2.0
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            x, y = a[-lag:], b[:len(b) + lag]
        elif lag > 0:
            x, y = a[:-lag], b[lag:]
        else:
            x, y = a, b
        n = min(len(x), len(y))
        if n < 8:
            continue
        c = float(np.mean(x[:n] * y[:n]))
        if c > best_c:
            best_c, best_lag = c, lag
    return best_c, best_lag

_records = []
for _sid in tqdm(_paired, desc="verify alignment"):
    _h = load_helium(_helium[_sid])
    _wav, _sr = sf.read(str(_wavs[_sid]))
    if _wav.ndim > 1:
        _wav = _wav.mean(axis=1)
    _wav = np.asarray(_wav, dtype=np.float32)
    T = int(_h.shape[0])
    dur = len(_wav) / float(_sr)
    dur_ok = abs(dur - T * STEP_SEC) <= max(0.16, 0.12 * T * STEP_SEC)
    # per-step audio energy envelope (resample wav length onto T steps)
    idx = np.linspace(0, len(_wav), T + 1).astype(int)
    step_rms = np.array([np.sqrt(np.mean(_wav[idx[i]:idx[i + 1]] ** 2) + 1e-12) for i in range(T)])
    norms = _h.norm(dim=-1).numpy()
    corr, lag = xcorr_lag(norms, step_rms, max_lag=12)
    voiced = step_rms > max(0.01, 0.08 * float(step_rms.max()))
    # Structural criterion only: the reply duration must match the step count.
    # The helium-norm vs audio-RMS envelope correlation is a WEAK proxy
    # (hidden-state norm is not loudness) -- diagnostic only, never a gate.
    # The strong content-level check is the trained frontend-adapter
    # regression in a later cell (helium -> w2v prediction vs stored targets).
    ok = bool(dur_ok)
    _records.append(dict(
        sample_id=_sid, T=T, dur=round(dur, 3), energy_corr=round(corr, 4), energy_lag=lag,
        voiced_frac=round(float(voiced.mean()), 4), dur_ok=dur_ok, aligned=ok,
        helium_path=str(_helium[_sid]), wav_path=str(_wavs[_sid]),
        adapter_final_path=str(_adapt_final.get(_sid, "")),
        adapter_frontend_path=str(_adapt_front.get(_sid, "")),
    ))

import pandas as pd
valid_df = pd.DataFrame(_records)
valid_df.to_csv(VALID_INDEX_CSV, index=False)
_n_ok = int(valid_df.aligned.sum())
log("duration alignment verified: %d/%d samples OK (%.1f%%)" % (_n_ok, len(valid_df), 100.0 * _n_ok / len(valid_df)))
_lag_counts = valid_df.energy_lag.value_counts().sort_index()
log("median energy corr = %.3f | lag histogram: %s" % (float(valid_df.energy_corr.median()), _lag_counts.to_dict()))
_mode_lag = int(_lag_counts.idxmax())
_mode_frac = float(_lag_counts.max()) / len(valid_df)
SYSTEMATIC_LAG_STEPS = _mode_lag if _mode_frac >= 0.5 else 0
if SYSTEMATIC_LAG_STEPS:
    log("systematic helium<->audio offset: %+d steps (%.0f%% of samples); recorded in the results JSON" % (SYSTEMATIC_LAG_STEPS, 100.0 * _mode_frac))
else:
    log("no dominant envelope lag -- weak proxy, as expected; structural duration alignment governs")
if _n_ok < 0.95 * len(valid_df):
    log("WARNING: %d samples have mismatched durations and are excluded from probing." % (len(valid_df) - _n_ok))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(valid_df.energy_corr, bins=40); axes[0].set_title("energy xcorr peak (norms vs audio RMS)")
axes[1].hist(valid_df.energy_lag, bins=np.arange(-12.5, 13.5)); axes[1].set_title("energy xcorr lag (steps)")
axes[2].hist(valid_df["T"], bins=40); axes[2].set_title("sample length (steps; 100 = 8 s)")
fig.tight_layout(); fig.savefig(str(PLOTS_DIR / "alignment_summary.png"), dpi=110); plt.close(fig)
log("saved %s" % (PLOTS_DIR / "alignment_summary.png"))

## Offline teacher labeling (vocal + semantic, kept separate)

Two offline teachers label each reply:

- **Vocal-expression teacher** -- `SpeechEmotionClassifier` from the repository's own `IMTalker/auto_emotion.py` (the same `emotion2vec_plus_base` family the live system uses on the user microphone). Labels *how the reply sounds*. We bypass the repo's argmax-only `predict` and keep the **full score vector** for calibration analysis and Phase 2 soft-label training.
- **Semantic-content teacher** -- Whisper ASR transcript + `emotion-english-distilroberta-base` text classifier. Labels *what the reply says*.

Both label sets, full score vectors, and transcripts are stored **per sample, side by side**. Nothing is fused yet, and the scenario/intended emotion (if any) never becomes a label.

In [ ]:
# ===================== Offline teacher models (load) =========================
if str(IMTALKER_DIR) not in sys.path:
    sys.path.insert(0, str(IMTALKER_DIR))
from auto_emotion import SpeechEmotionClassifier, SER_LABEL_TO_EMOTION_ID

log("loading vocal teacher %s on %s ..." % (AUDIO_TEACHER_MODEL, TEACHER_DEVICE))
torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
audio_teacher = SpeechEmotionClassifier(AUDIO_TEACHER_MODEL, device=TEACHER_DEVICE, half=False, disgust_map=DISGUST_MAP)
TEACHER_VRAM_GB = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
log("vocal teacher ready (%.2f GB VRAM)" % TEACHER_VRAM_GB)

import torchaudio

def _english_part(label):
    parts = [p.strip() for p in str(label).split("/")]
    for p in parts:
        if p and all(ord(c) < 128 for c in p):
            return p.lower()
    return parts[0].lower()

EMOTION2VEC_TO_ID = {  # normalized English label -> avatar emotion id
    "angry": 3, "anger": 3, "happy": 1, "sad": 2, "neutral": 0,
    "surprised": 4, "surprise": 4, "fearful": 5, "fear": 5, "disgusted": DISGUST_MAP,
}

def audio_teacher_scores(pcm_24k):
    wav = torch.from_numpy(np.asarray(pcm_24k, np.float32).reshape(1, -1))
    wav = torchaudio.functional.resample(wav, 24000, 16000)
    pcm16 = wav[0].numpy()
    try:
        results = audio_teacher.model.generate(pcm16, fs=16000, granularity="utterance", extract_embedding=False)
    except Exception:
        import soundfile as _sf, tempfile as _tmp
        with _tmp.NamedTemporaryFile(suffix=".wav", delete=False) as f:
            _sf.write(f.name, pcm16, 16000)
            tmp_path = f.name
        try:
            results = audio_teacher.model.generate(tmp_path, fs=16000, granularity="utterance", extract_embedding=False)
        finally:
            os.unlink(tmp_path)
    r = results[0]
    labels = [str(x) for x in (r.get("labels") or [])]
    scores = [float(x) for x in (r.get("scores") or [])]
    return dict(zip(labels, scores))

def map_scores_to_6(scores_dict, mapping):
    out = np.zeros(NUM_EMOTIONS, dtype=np.float64)
    for label, score in scores_dict.items():
        key = _english_part(label)
        eid = mapping.get(key)
        if eid is None:
            continue  # unmapped mass (e.g. 'other', 'unknown') is dropped, then renormalized
        out[eid] = max(out[eid], float(score))
    s = out.sum()
    return out / s if s > 0 else out + 1.0 / NUM_EMOTIONS

if RUN_TEXT_TEACHER:
    from transformers import pipeline as _hf_pipeline
    log("loading ASR %s + text teacher %s ..." % (ASR_MODEL, TEXT_TEACHER_MODEL))
    _dev = 0 if TEACHER_DEVICE == "cuda" else -1
    asr_pipe = _hf_pipeline("automatic-speech-recognition", model=ASR_MODEL, device=_dev)
    text_teacher = _hf_pipeline("text-classification", model=TEXT_TEACHER_MODEL, top_k=None, device=_dev)
    TEXT_LABEL_TO_ID = {"anger": 3, "joy": 1, "neutral": 0, "sadness": 2, "surprise": 4, "fear": 5, "disgust": DISGUST_MAP}
    log("text teacher ready")
else:
    asr_pipe = None
    text_teacher = None
    log("text teacher DISABLED (RUN_TEXT_TEACHER=False)")

def transcribe(pcm_24k):
    wav = torchaudio.functional.resample(torch.from_numpy(pcm_24k.reshape(1, -1)), 24000, 16000)[0].numpy()
    return str(asr_pipe(wav)["text"]).strip()

def text_teacher_scores(text):
    if not text:
        return {}
    out = text_teacher(text[:512])
    rows = out[0] if isinstance(out, list) and out and isinstance(out[0], list) else out
    return {str(r["label"]).lower(): float(r["score"]) for r in rows}

log("teachers loaded.")

In [ ]:
# ================ Vocal-expression labeling (audio teacher) ==================
# Resume-safe: rows already present in labels.csv are skipped.
import pandas as pd

LABEL_COLS = [
    "sample_id",
    "vocal_label_id", "vocal_label", "vocal_conf", "vocal_scores_json",
    "semantic_label_id", "semantic_label", "semantic_conf", "semantic_scores_json", "transcript",
    "expressed_label_id", "expressed_label", "agreement", "review_needed", "corrected",
]
if LABELS_CSV.exists():
    labels_df = pd.read_csv(LABELS_CSV)
else:
    labels_df = pd.DataFrame(columns=LABEL_COLS)
_done = set(labels_df.sample_id.astype(str)) if len(labels_df) else set()

_rows = []
_flush = 0
for _sid in tqdm(_paired, desc="vocal labels"):
    if _sid in _done:
        continue
    _wav, _sr = sf.read(str(_wavs[_sid]))
    if _wav.ndim > 1:
        _wav = _wav.mean(axis=1)
    _wav = np.asarray(_wav, dtype=np.float32)
    _scores = audio_teacher_scores(_wav)
    _six = map_scores_to_6(_scores, EMOTION2VEC_TO_ID)
    _eid = int(_six.argmax())
    _rows.append({
        "sample_id": _sid,
        "vocal_label_id": _eid, "vocal_label": EMOTION_NAMES[_eid],
        "vocal_conf": round(float(_six[_eid]), 4),
        "vocal_scores_json": json.dumps([round(float(x), 4) for x in _six]),
    })
    _flush += 1
    if _flush % 50 == 0:
        labels_df = pd.concat([labels_df, pd.DataFrame(_rows)], ignore_index=True)
        labels_df.to_csv(LABELS_CSV, index=False)
        _rows, _done = [], set(labels_df.sample_id.astype(str))
        log("  flushed (%d labeled)" % len(labels_df))
if _rows:
    labels_df = pd.concat([labels_df, pd.DataFrame(_rows)], ignore_index=True)
labels_df.to_csv(LABELS_CSV, index=False)
log("vocal labeling complete: %d samples" % len(labels_df))
print("vocal label distribution:")
print(labels_df.vocal_label.value_counts().to_string())
print("median vocal confidence: %.3f" % float(labels_df.vocal_conf.median()))

In [ ]:
# ================ Semantic-content labeling (ASR + text teacher) =============
if RUN_TEXT_TEACHER:
    labels_df = pd.read_csv(LABELS_CSV)
    _need = labels_df[labels_df.semantic_label.isna()].copy()
    _flush = 0
    for _i, _row in tqdm(_need.iterrows(), total=len(_need), desc="semantic labels"):
        _sid = str(_row.sample_id)
        _wav, _sr = sf.read(str(_wavs[_sid]))
        if _wav.ndim > 1:
            _wav = _wav.mean(axis=1)
        _wav = np.asarray(_wav, dtype=np.float32)
        try:
            _text = transcribe(_wav)
        except Exception as e:
            _text = ""
            log("ASR failed for %s: %s" % (_sid, e))
        _tscores = text_teacher_scores(_text) if _text else {}
        _six = map_scores_to_6(_tscores, TEXT_LABEL_TO_ID) if _tscores else np.full(NUM_EMOTIONS, np.nan)
        _m = labels_df.sample_id.astype(str) == _sid
        labels_df.loc[_m, "transcript"] = _text
        if _tscores:
            _eid = int(np.nanargmax(_six))
            labels_df.loc[_m, "semantic_label_id"] = _eid
            labels_df.loc[_m, "semantic_label"] = EMOTION_NAMES[_eid]
            labels_df.loc[_m, "semantic_conf"] = round(float(_six[_eid]), 4)
            labels_df.loc[_m, "semantic_scores_json"] = json.dumps([round(float(x), 4) for x in _six])
        _flush += 1
        if _flush % 50 == 0:
            labels_df.to_csv(LABELS_CSV, index=False)
            log("  flushed semantic (%d/%d)" % (_flush, len(_need)))
    labels_df.to_csv(LABELS_CSV, index=False)
    log("semantic labeling complete.")
    print("semantic label distribution:")
    print(labels_df.semantic_label.value_counts(dropna=False).to_string())
else:
    log("semantic labeling skipped (RUN_TEXT_TEACHER=False); expressed labels will fall back to vocal.")

In [ ]:
# ============== Label fusion + human review mechanism ========================
# REQUIREMENT: preserve vocal and semantic labels separately; the fused
# 'expressed' label is derived transparently; disagreements and low-confidence
# rows go to a human review queue; scenario/intended emotion is never a label.
FUSION_POLICY = "vocal_wins"   # "vocal_wins" | "semantic_wins" | "confidence" | "drop"
LOW_CONF = 0.5

labels_df = pd.read_csv(LABELS_CSV)
_has_sem = labels_df.semantic_label_id.notna()
_agree = _has_sem & (labels_df.vocal_label_id == labels_df.semantic_label_id)

labels_df["agreement"] = np.where(~_has_sem, "no_semantic", np.where(_agree, "agree", "disagree"))
labels_df["review_needed"] = False

def _fuse(row):
    v = int(row.vocal_label_id)
    if pd.isna(row.semantic_label_id):
        return v, float(row.vocal_conf) < LOW_CONF
    s = int(row.semantic_label_id)
    if v == s:
        conf = (float(row.vocal_conf) + float(row.semantic_conf)) / 2.0
        return v, conf < LOW_CONF
    # disagreement
    if FUSION_POLICY == "semantic_wins":
        return s, True
    if FUSION_POLICY == "confidence":
        return (v if float(row.vocal_conf) >= float(row.semantic_conf) else s), True
    if FUSION_POLICY == "drop":
        return -1, True
    return v, True  # vocal_wins

_fused = labels_df.apply(_fuse, axis=1, result_type="expand")
labels_df["expressed_label_id"] = _fused[0].astype(int)
labels_df["review_needed"] = labels_df["review_needed"] | _fused[1].astype(bool)
labels_df["expressed_label"] = labels_df.expressed_label_id.map(lambda i: EMOTION_NAMES.get(int(i), "dropped"))
labels_df["corrected"] = labels_df["corrected"].fillna(False)
labels_df.to_csv(LABELS_CSV, index=False)

_review = labels_df[labels_df.review_needed].copy()
_review.to_csv(REVIEW_QUEUE_CSV, index=False)
log("fusion done (%s): agreement=%s" % (FUSION_POLICY, labels_df.agreement.value_counts().to_dict()))
log("review queue: %d rows -> %s" % (len(_review), REVIEW_QUEUE_CSV))
print("expressed label distribution (dropped=-1 excluded from training):")
print(labels_df.expressed_label.value_counts().to_string())
print()
print("Human review workflow:")
print("  1. Open review_queue.csv (disagreements + low-confidence rows).")
print("  2. Listen to the reply WAVs (paths in valid_index.csv) and fill corrections.csv")
print("     with columns: sample_id, expressed_label, vocal_label, semantic_label, note")
print("  3. Re-run the NEXT cell to apply corrections (idempotent).")

In [ ]:
# ===================== Apply human corrections (idempotent) ==================
def apply_corrections(path=CORRECTIONS_CSV):
    if not Path(path).exists():
        log("no corrections file at %s -- nothing to apply" % path)
        return 0
    corr = pd.read_csv(path)
    labels_df = pd.read_csv(LABELS_CSV)
    n_applied = 0
    for _, r in corr.iterrows():
        m = labels_df.sample_id.astype(str) == str(r.sample_id)
        if not m.any():
            continue
        for col, idcol in (("expressed_label", "expressed_label_id"), ("vocal_label", "vocal_label_id"), ("semantic_label", "semantic_label_id")):
            val = r.get(col)
            if isinstance(val, str) and val.strip() and val.strip().lower() != "nan":
                name = val.strip().lower()
                if name in EMOTION_IDS:
                    labels_df.loc[m, col] = name
                    labels_df.loc[m, idcol] = EMOTION_IDS[name]
        labels_df.loc[m, "review_needed"] = False
        labels_df.loc[m, "corrected"] = True
        n_applied += 1
    labels_df.to_csv(LABELS_CSV, index=False)
    labels_df[labels_df.review_needed].to_csv(REVIEW_QUEUE_CSV, index=False)
    log("applied %d corrections; review queue now %d rows" % (n_applied, int(labels_df.review_needed.sum())))
    return n_applied

apply_corrections()
labels_df = pd.read_csv(LABELS_CSV)
TRAINABLE_DF = labels_df[(labels_df.expressed_label_id >= 0)].copy()
log("trainable samples (expressed labels): %d of %d" % (len(TRAINABLE_DF), len(labels_df)))

## Representation extraction (the candidate input menu)

We extract and cache every candidate network input so probes can be re-run cheaply:

| Feature | Shape | Source |
|---|---|---|
| `hidden_mean` | [4096] | voiced-masked mean of per-step Helium hidden states |
| `hidden_meanstd` | [8192] | masked mean concat per-dim std |
| `hidden_steps` | [T, 4096] | full per-step sequence (temporal heads) |
| `adapter_mean` | [768] | frozen Wav2Vec final-layer target, mean-pooled (dataset's `w2v_final_50hz`) |
| `acoustic_stats` | [14] | DSP envelope stats from the reply WAV |
| `mimi_codes` | [8, T] | Mimi re-encoding of the reply WAV (audio-token-only baseline; optional, needs gated Mimi asset) |

Features are cached per sample in `features/`; re-running skips cached samples.

In [ ]:
# ======================= Representation extraction ===========================
valid_df = pd.read_csv(VALID_INDEX_CSV)
_valid_ok = valid_df[valid_df.aligned].copy()
log("extracting features for %d aligned samples" % len(_valid_ok))

# --- optional Mimi re-encoder ---
_mimi = None
if RUN_MIMI_CODES:
    try:
        from moshi.models import loaders as _ld
        _mimi_path = hf_hub_download("nvidia/personaplex-7b-v1", "tokenizer-e351c8d8-checkpoint125.safetensors",
                                     token=os.environ.get("HF_TOKEN") or None)
        _mimi = _ld.get_mimi(_mimi_path, device="cuda" if torch.cuda.is_available() else "cpu")
        _mimi.eval()
        log("Mimi loaded for audio-token re-encoding (%s)" % _mimi_path)
    except Exception as e:
        log("Mimi unavailable (%s). mimi_codes features will be skipped. Provide HF_TOKEN with access to the gated nvidia/personaplex-7b-v1 repo to enable." % str(e)[:200])
        RUN_MIMI_CODES = False

def acoustic_features(wav):
    x = torch.from_numpy(wav.reshape(1, -1))
    frame = 2048
    hop = 1024
    n_frames = max(1, (x.shape[1] - frame) // hop + 1)
    frames = x.unfold(1, frame, hop)
    rms = frames.pow(2).mean(-1).sqrt()[0]
    zcr = ((frames[:, :, 1:] * frames[:, :, :-1]) < 0).float().mean(-1)[0]
    spec = torchaudio.transforms.SpectralCentroid(24000, n_fft=frame, hop_length=hop)(x)[0]
    magspec = torchaudio.transforms.Spectrogram(n_fft=frame, hop_length=hop, power=1.0)(x)[0]  # [freq, time]
    # spectral rolloff: lowest bin holding 85% of cumulative magnitude.
    # (torchaudio has no SpectralRolloff transform -- computed manually.)
    _cum = magspec.cumsum(0)
    _roll_bins = (_cum >= 0.85 * _cum[-1].clamp_min(1e-8)).float().argmax(0)
    roll = _roll_bins.float() * (24000.0 / frame)  # bin index -> Hz
    mag = magspec.mean(0)
    flux = torch.diff(mag).abs()
    def st(t):
        return [float(t.mean()), float(t.std()), float(torch.quantile(t, 0.9))]
    feats = st(rms) + st(zcr)[:2] + st(spec)[:2] + st(roll)[:2] + [float(flux.mean()), float(flux.std()), float(rms.max())]
    return np.asarray(feats, dtype=np.float32)  # [14]

@torch.no_grad()
def mimi_encode(wav24k):
    x = torch.from_numpy(wav24k.reshape(1, 1, -1)).to(next(_mimi.parameters()).device)
    codes = _mimi.encode(x)  # [1, K, T]
    return codes[0].cpu()    # [K, T]

_n_new, _n_cached, _n_codefail = 0, 0, 0
for _, _r in tqdm(_valid_ok.iterrows(), total=len(_valid_ok), desc="features"):
    _sid = str(_r.sample_id)
    _out = FEAT_DIR / (_sid + ".pt")
    if _out.exists():
        _n_cached += 1
        continue
    _h = load_helium(Path(_r.helium_path))
    _wav, _sr = sf.read(str(_r.wav_path))
    if _wav.ndim > 1:
        _wav = _wav.mean(axis=1)
    _wav = np.asarray(_wav, dtype=np.float32)
    T = _h.shape[0]
    idx = np.linspace(0, len(_wav), T + 1).astype(int)
    step_rms = np.array([np.sqrt(np.mean(_wav[idx[i]:idx[i + 1]] ** 2) + 1e-12) for i in range(T)])
    voiced = torch.from_numpy(step_rms > max(0.01, 0.08 * float(step_rms.max())))
    hv = _h[voiced] if voiced.any() else _h
    feat = {
        "hidden_mean": hv.mean(0),
        "hidden_meanstd": torch.cat([hv.mean(0), hv.std(0) + 1e-6]),
        "hidden_steps": _h.half(),
        "voiced": voiced,
        "acoustic_stats": torch.from_numpy(acoustic_features(_wav)),
    }
    if isinstance(_r.adapter_final_path, str) and _r.adapter_final_path:
        _af = torch.load(_r.adapter_final_path, map_location="cpu")
        if isinstance(_af, dict):
            _af = next(iter(_af.values()))
        _af = _af[0] if _af.ndim == 3 else _af
        feat["adapter_mean"] = _af.float().mean(0)
    if _mimi is not None:
        try:
            feat["mimi_codes"] = mimi_encode(_wav)
        except Exception as e:
            _n_codefail += 1
            if _n_codefail <= 3:
                log("mimi encode failed for %s: %s" % (_sid, str(e)[:150]))
    torch.save(feat, _out)
    _n_new += 1
log("features ready: %d new, %d cached, %d mimi-code failures" % (_n_new, _n_cached, _n_codefail))
if _mimi is not None and torch.cuda.is_available():
    log("peak VRAM so far: %.2f GB" % (torch.cuda.max_memory_allocated() / 1e9))

## Leakage-safe splits + probe evaluation protocol

**Split rule:** groups sharing a filename stem (same source utterance/session) never cross train/val/test. Note the limitation: this dataset was generated with PersonaPlex's own voice, so *speaker* independence cannot be tested here -- that is what Phase 1's multi-voice collection is for. The split here only guards against near-duplicate utterance leakage.

**Targets (all three evaluated separately):**

- `vocal` -- audio-teacher labels (how it sounds)
- `semantic` -- text-teacher labels (what it says)
- `expressed` -- fused labels (what it actually expressed; the Phase 2 target)

**Metrics:** accuracy, macro F1, per-class F1, confusion matrix, plus an auxiliary binary neutral-vs-emotional score. Linear probes use the paper-style recipe (StandardScaler -> PCA(30) -> class-weighted logistic regression). Neural probes (GRU / tiny Transformer on per-step sequences) follow in the next cells.

In [ ]:
# ==================== Leakage-safe splits ====================================
from sklearn.model_selection import GroupShuffleSplit
import re

def group_key(sid):
    m = re.match(r"^(.*?)[_\-]?\d+$", str(sid))
    return (m.group(1) if m and m.group(1) else str(sid))

labels_df = pd.read_csv(LABELS_CSV)
SPLIT_CSV = PHASE0_DIR / "split_assignment.csv"
if SPLIT_CSV.exists():
    split_df = pd.read_csv(SPLIT_CSV)
    log("loaded existing split assignment (%d rows)" % len(split_df))
else:
    _sids = valid_df[valid_df.aligned].sample_id.astype(str).tolist()
    _groups = np.array([group_key(s) for s in _sids])
    gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=SEED)
    tr_val_idx, te_idx = next(gss1.split(_sids, groups=_groups))
    _rest = [(_sids[i], _groups[i]) for i in tr_val_idx]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION / (1 - TEST_FRACTION), random_state=SEED)
    tr_i, va_i = next(gss2.split([r[0] for r in _rest], groups=np.array([r[1] for r in _rest])))
    _assign = {}
    for i in tr_i:
        _assign[_rest[i][0]] = "train"
    for i in va_i:
        _assign[_rest[i][0]] = "val"
    for i in te_idx:
        _assign[_sids[i]] = "test"
    split_df = pd.DataFrame({"sample_id": list(_assign), "split": list(_assign.values()), "group": [group_key(s) for s in _assign]})
    split_df.to_csv(SPLIT_CSV, index=False)
    log("new split assignment: %s" % split_df.split.value_counts().to_dict())

_merged = split_df.merge(labels_df[["sample_id", "vocal_label", "semantic_label", "expressed_label"]], on="sample_id", how="left")
for _t in ("vocal_label", "semantic_label", "expressed_label"):
    print("split class balance (%s):" % _t)
    print(_merged.groupby("split")[_t].value_counts(normalize=True).round(3).unstack(fill_value=0).to_string())
SPLITS = {s: set(split_df[split_df.split == s].sample_id.astype(str)) for s in ("train", "val", "test")}

In [ ]:
# ================= Linear probes (pooled features) ===========================
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def load_feature_matrix(sids, kind):
    xs = []
    for sid in sids:
        f = torch.load(FEAT_DIR / (sid + ".pt"), map_location="cpu")
        if kind == "hidden+adapter":
            xs.append(torch.cat([f["hidden_mean"], f.get("adapter_mean", torch.zeros(768))]).numpy())
        elif kind == "hidden+acoustic":
            xs.append(torch.cat([f["hidden_mean"], f["acoustic_stats"]]).numpy())
        else:
            xs.append(f[kind].numpy())
    return np.stack(xs)

def get_labels(sids, target):
    col = {"vocal": "vocal_label_id", "semantic": "semantic_label_id", "expressed": "expressed_label_id"}[target]
    m = labels_df.set_index(labels_df.sample_id.astype(str))[col]
    ys, keep = [], []
    for i, sid in enumerate(sids):
        y = m.get(sid, np.nan)
        ok = pd.notna(y) and int(y) >= 0
        keep.append(ok)
        ys.append(int(y) if ok else -1)
    return np.array(ys), np.array(keep)

LINEAR_REPS = ["hidden_mean", "hidden_meanstd", "adapter_mean", "acoustic_stats", "hidden+adapter", "hidden+acoustic"]
results_linear = {}
_sids_all = [s for s in sorted(SPLITS["train"] | SPLITS["val"] | SPLITS["test"])]
for _target in ("vocal", "semantic", "expressed"):
    results_linear[_target] = {}
    for _rep in LINEAR_REPS:
        _tr = sorted(SPLITS["train"] | SPLITS["val"])
        _te = sorted(SPLITS["test"])
        Xtr, Xte = load_feature_matrix(_tr, _rep), load_feature_matrix(_te, _rep)
        ytr, ktr = get_labels(_tr, _target)
        yte, kte = get_labels(_te, _target)
        Xtr, ytr = Xtr[ktr], ytr[ktr]
        Xte, yte = Xte[kte], yte[kte]
        if len(np.unique(ytr)) < 2 or len(yte) < 10:
            results_linear[_target][_rep] = None
            continue
        n_pca = int(min(PCA_COMPONENTS, Xtr.shape[1], len(Xtr) - 1))
        clf = make_pipeline(StandardScaler(), PCA(n_components=n_pca, random_state=SEED),
                            LogisticRegression(max_iter=3000, class_weight="balanced"))
        clf.fit(Xtr, ytr)
        pred = clf.predict(Xte)
        acc = accuracy_score(yte, pred)
        mf1 = f1_score(yte, pred, average="macro")
        emo = yte > 0
        bin_f1 = f1_score(emo, pred > 0) if emo.any() and (~emo).any() else float("nan")
        results_linear[_target][_rep] = dict(acc=round(acc, 4), macro_f1=round(mf1, 4), binary_neutral_vs_emo_f1=round(float(bin_f1), 4),
                                             n_test=int(len(yte)), confusion=confusion_matrix(yte, pred, labels=list(range(NUM_EMOTIONS))).tolist())
        log("linear[%s][%s]: acc=%.3f macroF1=%.3f" % (_target, _rep, acc, mf1))

import joblib
joblib.dump({"config": dict(pca=PCA_COMPONENTS, seed=SEED), "results": results_linear}, PROBES_DIR / "linear_probes.joblib")
print()
print(pd.DataFrame({t: {r: (v["macro_f1"] if v else None) for r, v in d.items()} for t, d in results_linear.items()}).round(3).to_string())

## Neural probes on per-step sequences

The linear probes test *pooled* information. The candidate live head, however, sees a **stream of per-step hidden states** (the live deque keeps the last 100 steps = 8 s plus lookahead). These probes pre-validate the exact head families Phase 2 will train:

- **MLP** on `hidden_mean` (does pooling lose emotion information?)
- **GRU** on `hidden_steps` (small recurrent readout)
- **Tiny Transformer** on `hidden_steps` (2-layer, 2-head self-attention readout)
- **Codes-GRU** on `mimi_codes` only (audio-token-only baseline; tests whether the *tokens themselves* carry emotion)
- **Fusion-GRU** on `hidden_steps` + `mimi_codes` (tests whether tokens add anything beyond hidden states)

All use class-weighted cross-entropy, AdamW, early stopping on validation macro-F1, and are evaluated on the same leakage-safe test split as the linear probes.

In [ ]:
# ============ Lightweight neural baselines (temporal readouts) ===============
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class MLPHead(nn.Module):
    def __init__(self, d_in=4096, hidden=512, n_cls=NUM_EMOTIONS):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, hidden), nn.GELU(), nn.Dropout(0.2), nn.Linear(hidden, n_cls))
    def forward(self, x, lengths=None, codes=None):
        return self.net(x)

class GRUHead(nn.Module):
    def __init__(self, d_in=4096, hidden=256, n_cls=NUM_EMOTIONS):
        super().__init__()
        self.proj = nn.Linear(d_in, hidden)
        self.gru = nn.GRU(hidden, hidden, batch_first=True)
        self.out = nn.Linear(hidden, n_cls)
    def forward(self, x, lengths=None, codes=None):
        h = self.proj(x)
        h, _ = self.gru(h)
        if lengths is not None:
            idx = (lengths - 1).clamp(min=0).view(-1, 1, 1).expand(-1, 1, h.shape[-1])
            h = h.gather(1, idx)[:, 0]
        else:
            h = h[:, -1]
        return self.out(h)

class TinyTransformerHead(nn.Module):
    def __init__(self, d_in=4096, d_model=256, n_heads=4, n_layers=2, n_cls=NUM_EMOTIONS, max_len=1024):
        super().__init__()
        self.proj = nn.Linear(d_in, d_model)
        self.pos = nn.Parameter(torch.zeros(1, max_len, d_model))
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4, dropout=0.1, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, n_layers)
        self.out = nn.Linear(d_model, n_cls)
    def forward(self, x, lengths=None, codes=None):
        h = self.proj(x) + self.pos[:, :x.shape[1]]
        pad_mask = None
        if lengths is not None:
            pad_mask = torch.arange(x.shape[1], device=x.device)[None, :] >= lengths[:, None]
        h = self.enc(h, src_key_padding_mask=pad_mask)
        if pad_mask is not None:
            m = (~pad_mask).float().unsqueeze(-1)
            h = (h * m).sum(1) / m.sum(1).clamp(min=1)
        else:
            h = h.mean(1)
        return self.out(h)

class CodesGRUHead(nn.Module):
    def __init__(self, n_books=8, card=2048, emb=32, hidden=256, n_cls=NUM_EMOTIONS):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(card, emb) for _ in range(n_books)])
        self.gru = nn.GRU(n_books * emb, hidden, batch_first=True)
        self.out = nn.Linear(hidden, n_cls)
    def forward(self, x=None, lengths=None, codes=None):
        e = torch.cat([emb(codes[:, i]) for i, emb in enumerate(self.embs)], dim=-1)
        h, _ = self.gru(e)
        idx = (lengths - 1).clamp(min=0).view(-1, 1, 1).expand(-1, 1, h.shape[-1])
        return self.out(h.gather(1, idx)[:, 0])

class FusionGRUHead(nn.Module):
    def __init__(self, d_in=4096, n_books=8, card=2048, emb=32, hidden=256, n_cls=NUM_EMOTIONS):
        super().__init__()
        self.proj = nn.Linear(d_in, hidden)
        self.embs = nn.ModuleList([nn.Embedding(card, emb) for _ in range(n_books)])
        self.gru = nn.GRU(hidden + n_books * emb, hidden, batch_first=True)
        self.out = nn.Linear(hidden, n_cls)
    def forward(self, x, lengths=None, codes=None):
        h = self.proj(x)
        e = torch.cat([emb(codes[:, i]) for i, emb in enumerate(self.embs)], dim=-1)
        T = min(h.shape[1], e.shape[1])
        h, _ = self.gru(torch.cat([h[:, :T], e[:, :T]], dim=-1))
        idx = (lengths.clamp(max=T) - 1).clamp(min=0).view(-1, 1, 1).expand(-1, 1, h.shape[-1])
        return self.out(h.gather(1, idx)[:, 0])

class SeqDataset(torch.utils.data.Dataset):
    def __init__(self, sids, target, use_codes=False):
        self.items = []
        m = labels_df.set_index(labels_df.sample_id.astype(str))
        col = {"vocal": "vocal_label_id", "semantic": "semantic_label_id", "expressed": "expressed_label_id"}[target]
        for sid in sids:
            f = torch.load(FEAT_DIR / (sid + ".pt"), map_location="cpu")
            y = m[col].get(sid, np.nan)
            if pd.isna(y) or int(y) < 0:
                continue
            codes = f.get("mimi_codes") if use_codes else None
            if use_codes and codes is None:
                continue
            self.items.append((f["hidden_steps"].float(), codes, int(y)))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]

def collate(batch):
    Ts = [b[0].shape[0] for b in batch]
    Tmax = max(Ts)
    xs = torch.zeros(len(batch), Tmax, 4096)
    lengths = torch.tensor(Ts)
    ys = torch.tensor([b[2] for b in batch])
    codes = None
    if batch[0][1] is not None:
        K = batch[0][1].shape[0]
        codes = torch.zeros(len(batch), K, Tmax, dtype=torch.long)
        for i, b in enumerate(batch):
            Tc = min(b[1].shape[1], Tmax)
            codes[i, :, :Tc] = b[1][:, :Tc]
    for i, b in enumerate(batch):
        xs[i, :Ts[i]] = b[0]
    return xs, lengths, codes, ys

def train_probe(model, train_sids, val_sids, target, use_codes=False, epochs=TORCH_PROBE_EPOCHS, bs=32, lr=3e-4, tag=""):
    tr_loader = torch.utils.data.DataLoader(SeqDataset(train_sids, target, use_codes), batch_size=bs, shuffle=True, collate_fn=collate, drop_last=False)
    va_loader = torch.utils.data.DataLoader(SeqDataset(val_sids, target, use_codes), batch_size=bs, shuffle=False, collate_fn=collate)
    if len(tr_loader.dataset) == 0:
        return None, None
    ys = torch.tensor([b[2] for b in tr_loader.dataset.items])
    counts = torch.bincount(ys, minlength=NUM_EMOTIONS).float()
    weights = (counts.sum() / counts.clamp(min=1)).clamp(max=10.0).to(DEVICE)
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best_f1, best_state, bad = -1.0, None, 0
    def evaluate(loader):
        model.eval()
        preds, gold = [], []
        with torch.no_grad():
            for xs, lengths, codes, y in loader:
                logits = model(xs.to(DEVICE), lengths.to(DEVICE), codes.to(DEVICE) if codes is not None else None)
                preds.append(logits.argmax(-1).cpu())
                gold.append(y)
        preds, gold = torch.cat(preds), torch.cat(gold)
        return f1_score(gold.numpy(), preds.numpy(), average="macro"), gold.numpy(), preds.numpy()
    for ep in range(epochs):
        model.train()
        for xs, lengths, codes, y in tr_loader:
            logits = model(xs.to(DEVICE), lengths.to(DEVICE), codes.to(DEVICE) if codes is not None else None)
            loss = F.cross_entropy(logits, y.to(DEVICE), weight=weights)
            opt.zero_grad(); loss.backward(); opt.step()
        vf1, _, _ = evaluate(va_loader)
        if vf1 > best_f1:
            best_f1, best_state, bad = vf1, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= 5:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    log("probe %s[%s]: best val macroF1=%.3f" % (tag, target, best_f1))
    return model, evaluate

log("neural probe machinery ready (device=%s)" % DEVICE)

In [ ]:
# =================== Run neural probes + test evaluation =====================
results_torch = {}
_trained_heads = {}
if RUN_TORCH_PROBES:
    for _target in ("vocal", "semantic", "expressed"):
        results_torch[_target] = {}
        _tr_sids = sorted(SPLITS["train"])
        _va_sids = sorted(SPLITS["val"])
        _te_sids = sorted(SPLITS["test"])
        _te_loader_cache = {}
        def test_metrics(model, use_codes):
            key = use_codes
            if key not in _te_loader_cache:
                _te_loader_cache[key] = torch.utils.data.DataLoader(
                    SeqDataset(_te_sids, _target, use_codes), batch_size=64, shuffle=False, collate_fn=collate)
            loader = _te_loader_cache[key]
            if len(loader.dataset) == 0:
                return None
            model.eval()
            preds, gold, probs = [], [], []
            with torch.no_grad():
                for xs, lengths, codes, y in loader:
                    logits = model(xs.to(DEVICE), lengths.to(DEVICE), codes.to(DEVICE) if codes is not None else None)
                    probs.append(torch.softmax(logits, -1).cpu())
                    preds.append(logits.argmax(-1).cpu())
                    gold.append(y)
            preds, gold, probs = torch.cat(preds).numpy(), torch.cat(gold).numpy(), torch.cat(probs).numpy()
            emo = gold > 0
            return dict(acc=round(float(accuracy_score(gold, preds)), 4),
                        macro_f1=round(float(f1_score(gold, preds, average="macro")), 4),
                        binary_neutral_vs_emo_f1=round(float(f1_score(emo, preds > 0)) if emo.any() and (~emo).any() else float("nan"), 4),
                        n_test=int(len(gold)), confusion=confusion_matrix(gold, preds, labels=list(range(NUM_EMOTIONS))).tolist(),
                        _probs=probs, _gold=gold)
        _specs = [
            ("mlp_hidden_mean", MLPHead(), False, "pooled"),
            ("gru_hidden_steps", GRUHead(), False, "seq"),
            ("tinyth_hidden_steps", TinyTransformerHead(), False, "seq"),
        ]
        if RUN_MIMI_CODES:
            _specs += [("codes_gru", CodesGRUHead(), True, "seq"), ("fusion_gru", FusionGRUHead(), True, "seq")]
        for _name, _model, _use_codes, _kind in _specs:
            if _kind == "pooled":
                # MLP on pooled features via the same seq pipeline (mean-pool first)
                _model = MLPHead()
                def pooled_forward(self, x, lengths=None, codes=None):
                    if lengths is not None:
                        m = (torch.arange(x.shape[1], device=x.device)[None, :] < lengths[:, None]).float().unsqueeze(-1)
                        x = (x * m).sum(1) / m.sum(1).clamp(min=1)
                    return self.net(x)
                import types as _types
                _model.forward = _types.MethodType(pooled_forward, _model)
            _trained, _ = train_probe(_model, _tr_sids, _va_sids, _target, use_codes=_use_codes, tag=_name)
            if _trained is None:
                results_torch[_target][_name] = None
                continue
            _m = test_metrics(_trained, _use_codes)
            if _m is not None:
                _trained_heads[(_target, _name)] = _trained
                results_torch[_target][_name] = {k: v for k, v in _m.items() if not k.startswith("_")}
                results_torch[_target][_name]["_probs"] = _m["_probs"]
                results_torch[_target][_name]["_gold"] = _m["_gold"]
                log("torch[%s][%s]: test acc=%.3f macroF1=%.3f" % (_target, _name, _m["acc"], _m["macro_f1"]))
    save_json({t: {n: ({k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in d.items() if not k.startswith("_")} if d else None)
                     for n, d in dd.items()} for t, dd in results_torch.items()},
              PROBES_DIR / "torch_probe_results.json")
    print()
    print(pd.DataFrame({t: {r: (v["macro_f1"] if v else None) for r, v in d.items()} for t, d in results_torch.items()}).round(3).to_string())
else:
    log("torch probes disabled (RUN_TORCH_PROBES=False)")

In [ ]:
# ================= Latency / memory benchmark ================================
# Budget context (live system): hidden states arrive at 12.5 Hz (80 ms/step);
# the live emotion decision runs on a ~960 ms chunk budget; a live head must be
# far below that (target < 10 ms GPU) so the renderer/FM path is untouched.
def bench_model(model, use_codes=False, iters=100, T=100):
    model = model.to(DEVICE).eval()
    xs = torch.randn(1, T, 4096, device=DEVICE)
    lengths = torch.tensor([T], device=DEVICE)
    codes = torch.randint(0, 2048, (1, 8, T), device=DEVICE) if use_codes else None
    with torch.no_grad():
        for _ in range(20):
            model(xs, lengths, codes)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        ts = []
        for _ in range(iters):
            t0 = time.perf_counter()
            model(xs, lengths, codes)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            ts.append((time.perf_counter() - t0) * 1e3)
    n_params = sum(p.numel() for p in model.parameters())
    return dict(params=n_params, params_mb=round(n_params * 4 / 1e6, 2),
                p50_ms=round(float(np.percentile(ts, 50)), 3), p95_ms=round(float(np.percentile(ts, 95)), 3))

latency = {}
if RUN_TORCH_PROBES and _trained_heads:
    for (_t, _n), _m in _trained_heads.items():
        if _t != "expressed":
            continue
        latency[_n] = bench_model(_m, use_codes=("codes" in _n or "fusion" in _n))
        log("latency[%s]: %s" % (_n, latency[_n]))
# linear-probe CPU latency (pooled path)
import joblib as _jl
if (PROBES_DIR / "linear_probes.joblib").exists():
    X = np.random.randn(1, 4096).astype(np.float32)
    _clf = make_pipeline(StandardScaler(), PCA(n_components=30, random_state=SEED), LogisticRegression(max_iter=100)).fit(np.random.randn(200, 4096), np.random.randint(0, 6, 200))
    ts = []
    for _ in range(100):
        t0 = time.perf_counter(); _clf.predict(X); ts.append((time.perf_counter() - t0) * 1e3)
    latency["linear_hidden_mean_cpu"] = dict(params="sklearn pipeline", p50_ms=round(float(np.percentile(ts, 50)), 3), p95_ms=round(float(np.percentile(ts, 95)), 3))
log("latency benchmark done")
print(json.dumps(latency, indent=2))
if DEVICE == "cuda":
    log("peak VRAM this session: %.2f GB" % (torch.cuda.max_memory_allocated() / 1e9))

In [ ]:
# ============================ Calibration (ECE) ==============================
def expected_calibration_error(probs, gold, n_bins=10):
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0:
            continue
        ece += (m.sum() / len(gold)) * abs(float((pred[m] == gold[m]).mean()) - float(conf[m].mean()))
    return float(ece)

calibration = {}
for _target, _dd in results_torch.items():
    calibration[_target] = {}
    for _name, _d in _dd.items():
        if not _d or "_probs" not in _d:
            continue
        calibration[_target][_name] = round(expected_calibration_error(_d["_probs"], _d["_gold"]), 4)
for _target, _dd in results_linear.items():
    calibration.setdefault(_target + "_linear", {})
log("ECE (lower is better): %s" % json.dumps(calibration, indent=2))

# reliability diagram for the best expressed model
if results_torch.get("expressed"):
    _best = max(((n, d) for n, d in results_torch["expressed"].items() if d and "_probs" in d),
                key=lambda nd: nd[1]["macro_f1"], default=None)
    if _best is not None:
        _probs, _gold = _best[1]["_probs"], _best[1]["_gold"]
        _conf, _pred = _probs.max(1), _probs.argmax(1)
        fig, ax = plt.subplots(figsize=(5, 5))
        bins = np.linspace(0, 1, 11)
        accs = [((_pred[( _conf > lo) & (_conf <= hi)] == _gold[(_conf > lo) & (_conf <= hi)]).mean() if ((_conf > lo) & (_conf <= hi)).sum() else np.nan) for lo, hi in zip(bins[:-1], bins[1:])]
        ax.plot([0, 1], [0, 1], "k--")
        ax.bar((bins[:-1] + bins[1:]) / 2, np.nan_to_num(accs), width=0.09, align="center", edgecolor="k")
        ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.set_title("reliability: %s (expressed)" % _best[0])
        fig.tight_layout(); fig.savefig(str(PLOTS_DIR / "calibration_reliability.png"), dpi=110); plt.close(fig)
        log("saved calibration_reliability.png")

In [ ]:
# ============================ Visualizations =================================
# confusion grids for the best model per target; F1 comparison bars; agreement heatmap
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (_target, _dd) in zip(axes, results_torch.items()):
    _cands = [(n, d) for n, d in _dd.items() if d]
    if not _cands:
        ax.axis("off"); continue
    _n, _d = max(_cands, key=lambda nd: nd[1]["macro_f1"])
    cm = np.array(_d["confusion"])
    ax.imshow(cm, cmap="Blues")
    ax.set_title("%s: %s (F1=%.2f)" % (_target, _n, _d["macro_f1"]))
    ax.set_xticks(range(NUM_EMOTIONS), [EMOTION_NAMES[i] for i in range(NUM_EMOTIONS)], rotation=45, ha="right")
    ax.set_yticks(range(NUM_EMOTIONS), [EMOTION_NAMES[i] for i in range(NUM_EMOTIONS)])
    for i in range(NUM_EMOTIONS):
        for j in range(NUM_EMOTIONS):
            if cm[i, j]:
                ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)
fig.tight_layout(); fig.savefig(str(PLOTS_DIR / "confusions_best.png"), dpi=110); plt.close(fig)

# agreement heatmap (vocal vs semantic teacher labels)
labels_df = pd.read_csv(LABELS_CSV)
_has = labels_df[labels_df.semantic_label_id.notna()]
if len(_has):
    cm2 = confusion_matrix(_has.vocal_label_id.astype(int), _has.semantic_label_id.astype(int), labels=list(range(NUM_EMOTIONS)))
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(cm2, cmap="Oranges")
    ax.set_title("teacher agreement: vocal (rows) vs semantic (cols)")
    ax.set_xticks(range(NUM_EMOTIONS), [EMOTION_NAMES[i] for i in range(NUM_EMOTIONS)], rotation=45, ha="right")
    ax.set_yticks(range(NUM_EMOTIONS), [EMOTION_NAMES[i] for i in range(NUM_EMOTIONS)])
    fig.tight_layout(); fig.savefig(str(PLOTS_DIR / "teacher_agreement.png"), dpi=110); plt.close(fig)

# F1 comparison bars (linear + torch, expressed target)
rows = []
for _rep, _d in results_linear.get("expressed", {}).items():
    if _d: rows.append(("linear:" + _rep, _d["macro_f1"]))
for _n, _d in results_torch.get("expressed", {}).items():
    if _d: rows.append(("torch:" + _n, _d["macro_f1"]))
if rows:
    fig, ax = plt.subplots(figsize=(10, 4))
    names, vals = zip(*sorted(rows, key=lambda r: r[1]))
    ax.barh(names, vals)
    ax.set_xlabel("macro F1 (test, expressed target)"); ax.set_title("expressed-emotion predictability by input+head")
    fig.tight_layout(); fig.savefig(str(PLOTS_DIR / "f1_comparison.png"), dpi=110); plt.close(fig)
log("plots saved to %s" % PLOTS_DIR)

In [ ]:
# ============== Phase 0 conclusions + results artifact =======================
# Recommendation logic: pick the best input+head for the EXPRESSED target
# (the Phase 2 training target), preferring hidden-only inputs on ties (they
# are free at runtime -- the live system already has hidden states), and
# requiring latency << the live budget.

def best_of(dd):
    cands = [(n, d) for n, d in dd.items() if d]
    if not cands:
        return None
    return max(cands, key=lambda nd: nd[1]["macro_f1"])

best_linear = {t: best_of(d) for t, d in results_linear.items()}
best_torch = {t: best_of(d) for t, d in results_torch.items()}

_reco_candidates = []
for _n, _d in (results_torch.get("expressed") or {}).items():
    if _d:
        _reco_candidates.append(("torch:" + _n, _d["macro_f1"], latency.get(_n, {}).get("p95_ms")))
for _n, _d in (results_linear.get("expressed") or {}).items():
    if _d:
        _reco_candidates.append(("linear:" + _n, _d["macro_f1"], latency.get("linear_hidden_mean_cpu", {}).get("p95_ms")))
_reco_candidates.sort(key=lambda r: (-(r[1] or 0), r[2] or 999))
_recommended = _reco_candidates[0][0] if _reco_candidates else None
if _recommended and "hidden" not in _recommended:
    _hidden_alts = [r for r in _reco_candidates if "hidden" in r[0]]
    if _hidden_alts and (_reco_candidates[0][1] - _hidden_alts[0][1]) < 0.02:
        _recommended = _hidden_alts[0][0]
        log("recommendation adjusted to hidden-only input (within 0.02 macro-F1 of best; hidden states are free at runtime)")

results = {
    "phase": 0,
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "repo_commit": REPO_COMMIT,
    "config": dict(dataset_hf_repo=DATASET_HF_REPO, full_dataset=FULL_DATASET, max_samples=MAX_SAMPLES,
                   audio_teacher=AUDIO_TEACHER_MODEL, text_teacher=TEXT_TEACHER_MODEL if RUN_TEXT_TEACHER else None,
                   asr=ASR_MODEL if RUN_TEXT_TEACHER else None, fusion_policy=FUSION_POLICY, seed=SEED),
    "dataset": dict(n_paired=len(_paired), n_aligned=int(valid_df.aligned.sum()),
                    median_energy_corr=float(valid_df.energy_corr.median()),
                    systematic_lag_steps=int(SYSTEMATIC_LAG_STEPS),
                    n_labeled=int(len(pd.read_csv(LABELS_CSV))),
                    n_trainable=int(len(labels_df[labels_df.expressed_label_id >= 0]))),
    "answers": {
        "vocal_from_hidden": (best_torch.get("vocal") or (None, None)),
        "semantic_from_hidden": (best_torch.get("semantic") or (None, None)),
        "expressed_best": (best_torch.get("expressed") or (None, None)),
    },
    "linear_results": {t: {r: v for r, v in d.items()} for t, d in results_linear.items()},
    "torch_results": {t: {r: ({k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in d.items() if not k.startswith("_")} if d else None) for r, d in dd.items()} for t, dd in results_torch.items()},
    "latency": latency,
    "calibration_ece": calibration,
    "recommendation": {
        "input_and_head": _recommended,
        "contract": dict(hidden_dim=4096, step_rate_hz=12.5, window_steps=100, lookahead_steps=6,
                         num_emotions=NUM_EMOTIONS, emotion_names=EMOTION_NAMES),
        "notes": [
            "hidden states are free at runtime (live system already captures them)",
            "speaker independence NOT testable with this single-voice dataset -- Phase 1 uses multiple voices",
            "vocal vs semantic predictability gap shows which information the head can actually learn",
        ],
    },
}
# strip numpy arrays for JSON safety
def _clean(o):
    if isinstance(o, dict):
        return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o
save_json(_clean(results), RESULTS_JSON)
log("wrote %s" % RESULTS_JSON)

# ---- human-readable report ----
lines = []
lines.append("# Phase 0 Report -- Helium Emotion Representation Validation")
lines.append("")
lines.append("- Repo commit: `%s`" % REPO_COMMIT)
lines.append("- Dataset: %d paired, %d aligned, %d labeled, %d trainable" % (len(_paired), int(valid_df.aligned.sum()), len(labels_df), len(labels_df[labels_df.expressed_label_id >= 0])))
lines.append("- Fusion policy: %s | Teachers: %s + %s" % (FUSION_POLICY, AUDIO_TEACHER_MODEL, TEXT_TEACHER_MODEL if RUN_TEXT_TEACHER else "(text teacher off)"))
lines.append("")
lines.append("## Core question: what can Helium hidden states predict?")
lines.append("")
lines.append("| Target | Best linear (macro F1) | Best neural (macro F1) |")
lines.append("|---|---|---|")
for t in ("vocal", "semantic", "expressed"):
    bl = best_linear.get(t); bt = best_torch.get(t)
    lines.append("| %s | %s | %s |" % (
        t,
        ("%s: %.3f" % (bl[0], bl[1]["macro_f1"])) if bl and bl[1] else "n/a",
        ("%s: %.3f" % (bt[0], bt[1]["macro_f1"])) if bt and bt[1] else "n/a"))
lines.append("")
lines.append("Interpretation: a large vocal-vs-semantic gap means hidden states encode prosodic/vocal emotion better than semantic content (or vice versa). Both matter for the unified head: the Phase 2 target is *expressed* emotion.")
lines.append("")
lines.append("## Recommendation for Phase 2")
lines.append("")
lines.append("- Input+head: `%s`" % _recommended)
lines.append("- Contract: hidden [T,4096] at 12.5 Hz, window=100 steps, 6 emotion classes matching generator/FM.py")
if latency:
    lines.append("- Latency: %s" % json.dumps({k: v for k, v in latency.items()}, default=str)[:600])
lines.append("")
lines.append("## Limitations")
lines.append("- Single-voice dataset: speaker independence untested (Phase 1 collects multiple voices).")
lines.append("- Teacher labels are proxies; disagreements are queued for human review (%d rows)." % int(labels_df.review_needed.sum()))
REPORT_MD.write_text("\n".join(lines), encoding="utf-8")
log("wrote %s" % REPORT_MD)
print("\n".join(lines))

## Phase 0 complete

Proceed to **Phase 1** (PersonaPlex emotional dataset collection) with `phase0_results.json` -- it tells Phase 2 which representation input and head family to train. If the expressed-emotion macro-F1 is very low across all representations, revisit label quality (apply human corrections and re-run probes) before investing in Phase 1 collection.

**Reproducibility:** all artifacts are under `/workspace/persona_emotion/phase0/` and every stage is resumable. Commit `phase0_report.md` and `phase0_results.json` to the repository for the record.